# HƯỚNG B — MULTIMODAL PRETRAINED (PaliGemma)
**Dự án:** Vietnamese Food VQA — Môn Học Sâu  
**Mô hình:** PaliGemma (google/paligemma-3b-pt-224)  

Notebook này thực hiện:
- **B1:** Zero-shot inference với PaliGemma
- **B2:** Fine-tune PaliGemma bằng LoRA/PEFT trên dataset ẩm thực Việt Nam
- **Evaluation:** So sánh B1 vs B2 dùng `metrics.py`

---
**FORMAT JSON DATASET** (đồng nhất với `dataset.py`):
```json
[
  {
    "image_id": "000010",
    "category": "am_thuc",
    "keyword":  "vit_quay",
    "qa_pairs": [
      {
        "question": "Day la mon gi?",
        "answer":   "Vit quay",
        "detailed_explanation": "Day la mon vit quay truyen thong..."
      }
    ]
  }
]
```
Đường dẫn ảnh: `img_dir / category / keyword / image_id.jpg`  
Field: `image_id`, `category`, `keyword`, `qa_pairs[].question`, `qa_pairs[].answer`


In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-80GB


## 0. Cài đặt thư viện

In [3]:
!pip install -q transformers peft accelerate bitsandbytes bert-score
!pip install -q Pillow tqdm sentencepiece protobuf
!pip install -q --upgrade torchao
from huggingface_hub import notebook_login
notebook_login()

## 1. Import & Cấu hình

In [ ]:
import os
import json
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Model
MODEL_ID      = "google/paligemma-3b-pt-224"
FINETUNED_DIR = "/content/paligemma_vqa_lora"

DATA_ROOT  = "/content/drive/MyDrive/Vietnamese_Cuisine_v1.0"
IMG_DIR    = os.path.join(DATA_ROOT, "images")          # -> images/am_thuc/{keyword}/
TRAIN_JSON = os.path.join(DATA_ROOT, "splits", "train.json")
VAL_JSON   = os.path.join(DATA_ROOT, "splits", "val.json")
TEST_JSON  = os.path.join(DATA_ROOT, "splits", "test.json")

DATA_DIR   = DATA_ROOT

# Hyperparameters
BATCH_SIZE     = 4
EPOCHS         = 5
LR             = 2e-4
MAX_NEW_TOKENS = 20
LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 2. Mount Google Drive & Load metrics.py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/metrics.py /content/metrics.py

import sys
sys.path.append('/content')
from metrics import evaluate_all, compare_configs, save_results, normalize_answer
print("metrics.py loaded thanh cong!")

Mounted at /content/drive
metrics.py loaded thanh cong!


## 3. Dataset Class cho PaliGemma

In [ ]:
class PaliGemmaVQADataset(Dataset):
    """
    Dataset cho PaliGemma — tương thích với cấu trúc JSON thực tế:
      item['image_path']              -> đường dẫn ảnh
      item['qa_pairs'][i]['question'] -> câu hỏi
      item['qa_pairs'][i]['answer']   -> câu trả lời ngắn
    Mỗi ảnh có 5 cặp Q&A -> flatten thành N*5 samples.
    """
    def __init__(self, json_path, img_dir, processor, is_train=True):
        self.processor = processor
        self.img_dir   = img_dir
        self.is_train  = is_train

        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        self.flat_data = []
        for item in raw_data:
            image_id = item['image_id']
            category = item.get('category', 'am_thuc')
            keyword  = item.get('keyword', '').replace(' ', '_')
            img_path = os.path.join(img_dir, category, keyword, f"{image_id}.jpg")

            for qa in item.get('qa_pairs', []):
                self.flat_data.append({
                    'image_path': img_path,
                    'question'  : qa['question'],
                    'answer'    : qa['answer'],
                })

        print(f"[Dataset] {os.path.basename(json_path)}: "
              f"{len(raw_data)} ảnh -> {len(self.flat_data)} cặp QA")

    def __len__(self):
        return len(self.flat_data)

    def __getitem__(self, idx):
        item = self.flat_data[idx]

        try:
            image = Image.open(item['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), color=0)

        question = item['question']
        answer   = item['answer']

        prompt = f"<image>\nTrả lời bằng tiếng Việt, ngắn gọn dưới 10 từ.\n{question}"

        if self.is_train:
            inputs = self.processor(
                text=prompt, images=image, suffix=answer,
                return_tensors='pt', truncation=True,
            )
        else:
            inputs = self.processor(
                text=prompt, images=image,
                return_tensors='pt', truncation=True,
            )

        inputs = {k: v.squeeze(0) for k, v in inputs.items() if isinstance(v, torch.Tensor)}
        inputs['answer']   = answer
        inputs['question'] = question
        return inputs


def collate_fn(batch):
    answers   = [b.pop('answer')   for b in batch]
    questions = [b.pop('question') for b in batch]

    pad_id = processor.tokenizer.pad_token_id

    def left_pad(tensors, padding_value):
        max_len = max(t.size(0) for t in tensors)
        return torch.stack([
            torch.cat([torch.full((max_len - t.size(0),), padding_value,
                                  dtype=t.dtype), t])
            for t in tensors
        ])

    input_ids      = left_pad([b['input_ids']      for b in batch], pad_id)
    attention_mask = left_pad([b['attention_mask'] for b in batch], 0)
    pixel_values   = torch.stack([b['pixel_values'] for b in batch])

    result = {
        'input_ids'     : input_ids,
        'attention_mask': attention_mask,
        'pixel_values'  : pixel_values,
        'answer'        : answers,
        'question'      : questions,
    }

    if 'labels' in batch[0]:
        result['labels'] = left_pad([b['labels'] for b in batch], -100)

    if 'token_type_ids' in batch[0]:
        result['token_type_ids'] = left_pad([b['token_type_ids'] for b in batch], 0)

    return result


## 4. Load Processor & Base Model

In [ ]:
print("Đang load PaliGemma Processor...")
processor = PaliGemmaProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

print("Đang load PaliGemma Model (bfloat16)...")
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
base_model.eval()
print(f"Load thành công! Tổng tham số: {sum(p.numel() for p in base_model.parameters()):,}")

Dang load PaliGemma Processor...


preprocessor_config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Load thanh cong! Tong tham so: 2,923,466,480


## 5. Hàm Inference dùng chung cho B1 và B2

In [8]:
def run_inference(model, processor, json_path, img_dir,
                  batch_size=2, max_new_tokens=MAX_NEW_TOKENS, desc="Inference"):
    dataset = PaliGemmaVQADataset(json_path, img_dir, processor, is_train=False)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         collate_fn=collate_fn, num_workers=0)

    all_predictions   = []
    all_ground_truths = []
    all_questions     = []

    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            answers   = batch.pop('answer')
            questions = batch.pop('question')

            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            pixel_values   = batch['pixel_values'].to(torch.bfloat16).to(DEVICE)

            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
            )

            new_tokens = generated_ids[:, input_ids.shape[1]:]
            decoded = processor.batch_decode(new_tokens, skip_special_tokens=True)

            all_predictions.extend([normalize_answer(d) for d in decoded])
            all_ground_truths.extend([[normalize_answer(a)] for a in answers])
            all_questions.extend(questions)

    return all_predictions, all_ground_truths, all_questions

## 6. B1 — Zero-shot

In [ ]:
print("=" * 50)
print("CẤU HÌNH B1: ZERO-SHOT")
print("=" * 50)

preds_b1, gts_b1, questions_b1 = run_inference(
    base_model, processor, TEST_JSON, IMG_DIR, batch_size=2, desc="B1 Zero-shot"
)

print("\n--- Mẫu kết quả B1 (5 mẫu dầu) ---")
for i in range(min(5, len(preds_b1))):
    print(f"Q   : {questions_b1[i]}")
    print(f"GT  : {gts_b1[i][0]}")
    print(f"Pred: {preds_b1[i]}")
    print()

CAU HINH B1: ZERO-SHOT
[Dataset] test.json: 126 anh -> 489 cap QA


B1 Zero-shot: 100%|██████████| 245/245 [04:35<00:00,  1.13s/it]


--- Mau ket qua B1 (5 mau dau) ---
Q:    Đây là gì?
GT:   giò thủ  |  Pred: bánh chuối

Q:    Hình ảnh này thể hiện điều gì?
GT:   một đĩa giò thủ đã được cắt sẵn  |  Pred: món ăn

Q:    Ý nghĩa văn hóa của giò thủ là gì?
GT:   sự sum họp may mắn và thịnh vượng  |  Pred: giò thủ là gì

Q:    Giò thủ có gì khác biệt so với các loại giò khác của Việt Nam?
GT:   nguyên liệu và cách chế biến  |  Pred: giò thủ là một loại giò được làm từ thịt heo thịt bò hoặc thịt lợn

Q:    Đây là gì?
GT:   giò thủ  |  Pred: thịt nguội



In [ ]:
# Đánh giá B1 với toàn bộ metrics
results_b1 = evaluate_all(
    predictions=preds_b1,
    ground_truths_list=gts_b1,
    questions=questions_b1,
    use_bertscore=True,
    use_llm_judge=False,
    config_name="B1",
)


  ĐÁNH GIÁ CẤU HÌNH: B1
  Số mẫu: 489
→ Tính VQA Accuracy (exact match)...
→ Tính BLEU...
→ Tính ROUGE-L...
→ Tính METEOR (đã sửa chunk counting)...
→ Tính BERTScore (vinai/phobert-base)...
  [Cảnh báo] BERTScore lỗi: 'vinai/phobert-base'

────────────────────────────────────────
  KẾT QUẢ [B1]
────────────────────────────────────────
  vqa_accuracy                 1.02
  num_samples                  489
  mode                         exact
  bleu_1                       6.22
  bleu_2                       2.98
  bleu_3                       0.38
  bleu_4                       0.2
  rouge_l                      8.56
  meteor                       7.4
────────────────────────────────────────



## 7. B2 — Fine-tune với LoRA

In [ ]:
print("Reload PaliGemma cho fine-tuning (bfloat16)...")
ft_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

for param in ft_model.model.vision_tower.parameters():
    param.requires_grad = False
for param in ft_model.model.multi_modal_projector.parameters():
    param.requires_grad = False

print("Đã freeze: vision_tower + multi_modal_projector")
print("Chỉ fine-tune: language_model (Gemma)")

Reload PaliGemma cho fine-tuning (bfloat16)...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

Da freeze: vision_tower + multi_modal_projector
Chi fine-tune: language_model (Gemma)


In [ ]:
# Cấu hình LoRA — áp dụng lên attention + FFN layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

ft_model = get_peft_model(ft_model, lora_config)
ft_model.print_trainable_parameters()

trainable params: 22,597,632 || all params: 2,946,064,112 || trainable%: 0.7670


In [ ]:
# Dataset & DataLoader
train_dataset = PaliGemmaVQADataset(TRAIN_JSON, IMG_DIR, processor, is_train=True)
val_dataset   = PaliGemmaVQADataset(VAL_JSON,   IMG_DIR, processor, is_train=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2
)

print(f"Train: {len(train_dataset)} mẫu | Val: {len(val_dataset)} mẫu")

[Dataset] train.json: 1001 anh -> 3879 cap QA
[Dataset] val.json: 125 anh -> 483 cap QA
Train: 3879 mau | Val: 483 mau


In [ ]:
# Optimizer & LR Scheduler
# Dùng AdamW + warmup (AdamW, weight_decay=1e-4)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, ft_model.parameters()),
    lr=LR,
    weight_decay=1e-4
)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

Total steps: 4850 | Warmup steps: 485


In [ ]:
def quick_validate(model, loader):
    """Tính VQA exact match accuracy nhanh trên val set."""
    model.eval()
    preds, gts = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating", leave=False):
            answers = batch.pop('answer')
            batch.pop('question')

            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            pixel_values   = batch['pixel_values'].to(torch.bfloat16).to(DEVICE)

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                token_type_ids=batch['token_type_ids'].to(DEVICE),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=3,
            )
            decoded = processor.batch_decode(
                gen_ids[:, input_ids.shape[1]:], skip_special_tokens=True
            )
            preds.extend([normalize_answer(d) for d in decoded])
            gts.extend([normalize_answer(a) for a in answers])

    correct = sum(p == g for p, g in zip(preds, gts))
    return correct / len(preds) * 100


# Training Loop
os.makedirs(FINETUNED_DIR, exist_ok=True)
best_val_acc  = 0.0
best_loss    = float('inf')
train_history = []

print("\n" + "=" * 50)
print("BẮT ĐẦU FINE-TUNE B2 — PaliGemma + LoRA")
print("=" * 50)

for epoch in range(EPOCHS):
    ft_model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in loop:
        batch.pop('answer')
        batch.pop('question')

        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        pixel_values   = batch['pixel_values'].to(torch.bfloat16).to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        token_type_ids = batch['token_type_ids'].to(DEVICE)

        outputs = ft_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            labels=labels,
            token_type_ids=token_type_ids,
        )

        loss = outputs.loss
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        loop.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    avg_loss = total_loss / len(train_loader)
    val_acc  = quick_validate(ft_model, val_loader)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%")
    train_history.append({"epoch": epoch+1, "loss": avg_loss, "val_acc": val_acc})

    if epoch == 0 or avg_loss < best_loss:
        best_loss = avg_loss
        ft_model.save_pretrained(FINETUNED_DIR)
        processor.save_pretrained(FINETUNED_DIR)
        print(f">> Saved best model (Loss: {avg_loss:.4f})")

print(f"\nFine-tuning hoàn tất! Best Loss: {best_loss:.4f}")


BAT DAU FINE-TUNE B2 — PaliGemma + LoRA


Epoch 1/5: 100%|██████████| 970/970 [10:01<00:00,  1.61it/s, loss=1.6547, lr=1.78e-04]


Epoch 1 | Loss: 1.3281 | Val Acc: 15.53%
  >> Saved best model (Loss: 1.3281)


Epoch 2/5: 100%|██████████| 970/970 [09:01<00:00,  1.79it/s, loss=1.1087, lr=1.33e-04]


Epoch 2 | Loss: 0.8140 | Val Acc: 17.18%
  >> Saved best model (Loss: 0.8140)


...

Epoch 3 | Loss: 0.5357 | Val Acc: 21.74%
  >> Saved best model (Loss: 0.5357)


Epoch 4/5: 100%|██████████| 970/970 [09:01<00:00,  1.79it/s, loss=0.1343, lr=4.44e-05]


Epoch 4 | Loss: 0.3307 | Val Acc: 21.33%
  >> Saved best model (Loss: 0.3307)


Epoch 5/5: 100%|██████████| 970/970 [09:02<00:00,  1.79it/s, loss=0.1684, lr=0.00e+00]


Epoch 5 | Loss: 0.1665 | Val Acc: 21.95%
  >> Saved best model (Loss: 0.1665)

Fine-tuning hoan tat! Best Loss: 0.1665


## 8. B2 — Load checkpoint & Inference

In [ ]:
print("Load fine-tuned model tu checkpoint...")
base_for_lora = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16,
)
ft_model_loaded = PeftModel.from_pretrained(base_for_lora, FINETUNED_DIR)
ft_model_loaded.eval()
print("Load thành công!")

Load fine-tuned model tu checkpoint...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/603 [00:00<?, ?it/s]

Load thanh cong!


In [ ]:
print("=" * 50)
print("CẤU HÌNH B2: FINE-TUNED (LoRA)")
print("=" * 50)

preds_b2, gts_b2, questions_b2 = run_inference(
    ft_model_loaded, processor, TEST_JSON, IMG_DIR, desc="B2 Fine-tuned"
)

print("\n--- Mẫu kết quả B2 (5 mẫu đầu) ---")
for i in range(min(5, len(preds_b2))):
    print(f"Q   : {questions_b2[i]}")
    print(f"GT  : {gts_b2[i][0]}")
    print(f"Pred: {preds_b2[i]}")
    print()

CAU HINH B2: FINE-TUNED (LoRA)
[Dataset] test.json: 126 anh -> 489 cap QA


B2 Fine-tuned: 100%|██████████| 245/245 [05:08<00:00,  1.26s/it]


--- Mau ket qua B2 (5 mau dau) ---
Q:    Đây là gì?
GT:   giò thủ  |  Pred: giò thủ

Q:    Hình ảnh này thể hiện điều gì?
GT:   một đĩa giò thủ đã được cắt sẵn  |  Pred: một đĩa chả giò được sắp xếp đẹp mắt

Q:    Ý nghĩa văn hóa của giò thủ là gì?
GT:   sự sum họp may mắn và thịnh vượng  |  Pred: sự sung túc và may mắn

Q:    Giò thủ có gì khác biệt so với các loại giò khác của Việt Nam?
GT:   nguyên liệu và cách chế biến  |  Pred: nguyên liệu và cách chế biến

Q:    Đây là gì?
GT:   giò thủ  |  Pred: giò thủ



In [19]:
results_b2 = evaluate_all(
    predictions=preds_b2,
    ground_truths_list=gts_b2,
    questions=questions_b2,
    use_bertscore=True,
    use_llm_judge=False,
    config_name="B2",
)


  ĐÁNH GIÁ CẤU HÌNH: B2
  Số mẫu: 489
→ Tính VQA Accuracy (exact match)...
→ Tính BLEU...
→ Tính ROUGE-L...
→ Tính METEOR (đã sửa chunk counting)...
→ Tính BERTScore (vinai/phobert-base)...
  [Cảnh báo] BERTScore lỗi: 'vinai/phobert-base'

────────────────────────────────────────
  KẾT QUẢ [B2]
────────────────────────────────────────
  vqa_accuracy                 23.31
  num_samples                  489
  mode                         exact
  bleu_1                       55.21
  bleu_2                       43.85
  bleu_3                       23.35
  bleu_4                       18.15
  rouge_l                      55.9
  meteor                       54.46
────────────────────────────────────────



## 9. So sánh B1 vs B2

In [ ]:
compare_configs([results_b1, results_b2])

DRIVE_OUTPUT = DATA_ROOT
save_results([results_b1, results_b2], f"{DRIVE_OUTPUT}/results_B.json")
print("Đã lưu results_B.json vào Drive!")


  BẢNG SO SÁNH CÁC CẤU HÌNH
Metric                                        B1                  B2
────────────────────────────────────────────────────────────────────
vqa_accuracy                                1.02             * 23.31
bleu_1                                      6.22             * 55.21
bleu_2                                      2.98             * 43.85
bleu_3                                      0.38             * 23.35
bleu_4                                       0.2             * 18.15
rouge_l                                     8.56              * 55.9
meteor                                       7.4             * 54.46

[metrics] Đã lưu kết quả → /content/drive/MyDrive/Vietnamese_Cuisine_v1.0/results_B.json
Da luu results_B.json vao Drive!


## 10. Phân tích lỗi (Error Analysis)

In [ ]:
def error_analysis(predictions, ground_truths, questions, config_name, top_n=10):
    errors = [
        (q, g[0], p)
        for q, g, p in zip(questions, ground_truths, predictions)
        if p != g[0]
    ]
    print(f"\n{'='*60}")
    print(f"  ERROR ANALYSIS [{config_name}]")
    print(f"  Tổng lỗi: {len(errors)}/{len(predictions)} "
          f"({len(errors)/len(predictions)*100:.1f}%)")
    print(f"{'='*60}")
    for i, (q, gt, pred) in enumerate(errors[:top_n]):
        print(f"[{i+1}] Q   :    {q}")
        print(f"    GT  :   {gt}")
        print(f"    Pred: {pred}")
        print()

error_analysis(preds_b1, gts_b1, questions_b1, "B1")
error_analysis(preds_b2, gts_b2, questions_b2, "B2")


  ERROR ANALYSIS [B1]
  Tong loi: 484/489 (99.0%)
[1] Q:    Đây là gì?
     GT:   giò thủ
     Pred: bánh chuối

[2] Q:    Hình ảnh này thể hiện điều gì?
     GT:   một đĩa giò thủ đã được cắt sẵn
     Pred: món ăn

[3] Q:    Ý nghĩa văn hóa của giò thủ là gì?
     GT:   sự sum họp may mắn và thịnh vượng
     Pred: giò thủ là gì

[4] Q:    Giò thủ có gì khác biệt so với các loại giò khác của Việt Nam?
     GT:   nguyên liệu và cách chế biến
     Pred: giò thủ là một loại giò được làm từ thịt heo thịt bò hoặc thịt lợn

[5] Q:    Đây là gì?
     GT:   giò thủ
     Pred: thịt nguội

[6] Q:    Hình ảnh này thể hiện điều gì?
     GT:   giò thủ và cách cắt lát của nó
     Pred: thực phẩm

[7] Q:    Ý nghĩa văn hóa của giò thủ là gì?
     GT:   sự sung túc may mắn và đoàn viên
     Pred: giò thủ là gì

[8] Q:    Tại sao giò thủ lại quan trọng trong văn hóa ẩm thực Việt Nam?
     GT:   do ý nghĩa văn hóa và hương vị đặc trưng
     Pred: 1

[9] Q:    Giò thủ có gì khác biệt so với các loại giò

## 11. Lưu predictions ra JSON (dùng cho demo_video.ipynb)

In [ ]:
def save_predictions(predictions, ground_truths, questions, path):
    """
    Lưu kết quả inference ra JSON.
    Format tương thích với:
      - load_predictions_from_json() trong metrics.py
      - load_preds() trong demo_video.ipynb
    """
    output = [
        {"question": q, "ground_truths": g, "prediction": p}
        for q, g, p in zip(questions, ground_truths, predictions)
    ]
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Da luu {len(output)} mau -> {path}")


save_predictions(preds_b1, gts_b1, questions_b1, f"{DATA_ROOT}/predictions_B1.json")
save_predictions(preds_b2, gts_b2, questions_b2, f"{DATA_ROOT}/predictions_B2.json")

print("\nHoàn tất Huớng B!")
print("Files đã lưu:")
print("predictions_B1.json  -> dùng cho demo_video.ipynb")
print("predictions_B2.json  -> dùng cho demo_video.ipynb")
print("results_B.json       -> dùng cho báo cáo + so sánh A vs B")

Da luu 489 mau -> /content/drive/MyDrive/Vietnamese_Cuisine_v1.0/predictions_B1.json
Da luu 489 mau -> /content/drive/MyDrive/Vietnamese_Cuisine_v1.0/predictions_B2.json

Hoan tat Huong B!
Files da luu:
  predictions_B1.json  -> dung cho demo_video.ipynb
  predictions_B2.json  -> dung cho demo_video.ipynb
  results_B.json       -> dung cho bao cao + so sanh A vs B


In [ ]:
import shutil
shutil.copytree(
    "/content/paligemma_vqa_lora",
    "/content/drive/MyDrive/paligemma_vqa_lora",
    dirs_exist_ok=True
)
print("Đã backup!")

Da backup!
